## **Importacion de Librerias necesarias**

In [1]:
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import os
import random

## **Parametros de la Curva**

In [2]:
p = 97
a = 2
b = 3
G = (3, 6)

print("Curva: y^2 = x^3 + {}x + {} mod {}".format(a,b,p))
print("Generador G =", G)

Curva: y^2 = x^3 + 2x + 3 mod 97
Generador G = (3, 6)


## **Funciones**

In [3]:
def inv_mod(k, p):
    return pow(k % p, -1, p)

def inv_mod(k, p):
    return pow(k % p, -1, p)

def suma(P, Q):
    if P is None:
        return Q
    if Q is None:
        return P
    
    x1, y1 = P
    x2, y2 = Q

    if x1 == x2 and (y1 + y2) % p == 0:
        return None

    if P != Q:
        m = ((y2 - y1) * inv_mod(x2 - x1, p)) % p
    else:
        if y1 == 0:
            return None
        m = ((3*x1*x1 + a) * inv_mod(2*y1, p)) % p

    x3 = (m*m - x1 - x2) % p
    y3 = (m*(x1 - x3) - y1) % p
    return (x3, y3)

def mult(k, P):
    R = None
    addend = P
    
    while k > 0:
        if k & 1:
            R = suma(R, addend)
        addend = suma(addend, addend)
        k >>= 1
    return R

## **Generacion de claves validas**

In [4]:
def generar_claves_validas():
    while True:
        priv = random.randint(1, 20)
        pub = mult(priv, G)
        if pub is not None:
            return priv, pub
        
alice_priv, alice_pub = generar_claves_validas()
bob_priv, bob_pub = generar_claves_validas()

print("Priv Alice:", alice_priv)
print("Pub Alice:", alice_pub)
print()
print("Priv Bob:", bob_priv)
print("Pub Bob:", bob_pub)

Priv Alice: 6
Pub Alice: (3, 6)

Priv Bob: 11
Pub Bob: (3, 6)


## **Calculo de Clave Compartida**

In [6]:
while True:
    compartido_alice = mult(alice_priv, bob_pub)
    compartido_bob = mult(bob_priv, alice_pub)
    
    if compartido_alice is not None and compartido_alice == compartido_bob:
        break
    else:
        alice_priv, alice_pub = generar_claves_validas()
        bob_priv, bob_pub = generar_claves_validas()

print("Shared Alice:", compartido_alice)
print("Shared Bob:", compartido_bob)
print("¿Coinciden?:", compartido_alice == compartido_bob)

Shared Alice: (3, 6)
Shared Bob: (3, 6)
¿Coinciden?: True


## **Derivacion de Clave (HKDF)**

In [9]:
compartido_x = compartido_alice[0]
compartido_en_bytes = compartido_x.to_bytes(2, 'big')

hkdf = HKDF(
    algorithm=hashes.SHA256(),
    length=32,
    salt=b'salt_demo',
    info=b'ECC Robust Demo',
)

key = hkdf.derive(compartido_en_bytes)
print("Clave derivada (32 bytes):", key)

Clave derivada (32 bytes): b'\x92\x99*\xa8\x94Z\x87Y\x07\x80\x02\xe7\xb7l\xac\xf5L\x7f\xa9\xc7\xdc\x00\xc42\xc9\xdb\x08\x05\x90\xb3\xde>'


## **Cifrado AES-GCM**

In [10]:
aes = AESGCM(key)
nonce = os.urandom(12)
mensaje_ingresado = input("Ingresa tu mensaje para encriptar")

In [12]:
print(mensaje_ingresado)

hola mundo


In [14]:
mensaje = mensaje_ingresado.encode()

print("Mensaje original:", mensaje)

cipher = aes.encrypt(nonce, mensaje, None)
print("Cipher:", cipher)

Mensaje original: b'hola mundo'
Cipher: b'\x1e\xc4\xa0(\x08\xf9Q\xa2\x94g\xa9\x0c\x98\x0e\x07\xaf\xf5\xa5\xcd\xe6\x0c\x1e\xe6\xed\xe0N'
